# Machine Learning – Practical 7: K-Means Clustering (Mall Customer Segmentation)

**Aim:** To segment mall customers into groups using K-Means clustering.

**Objective:**
- Standardise Annual Income and Spending Score
- Find the optimal number of clusters (Elbow method and Silhouette score)
- Train the final model and interpret the clusters

**Theory:** K-Means is an unsupervised algorithm that partitions data into K clusters by repeatedly assigning points to the nearest centroid and updating centroids. The Elbow method plots WCSS (within-cluster sum of squares) against K; the Silhouette score (−1 to 1) measures how well points fit their own cluster versus others.

## 1. Import Libraries

**Aim:** Import required libraries.

**Objective:** Load pandas, matplotlib and scikit-learn clustering tools.

**Theory:** KMeans and silhouette_score come from scikit-learn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("All libraries imported successfully!")

## 2. Load the Dataset

**Aim:** Load the Mall Customers data.

**Objective:** Read the CSV and inspect shape and columns.

**Theory:** The dataset has 200 customers with Gender, Age, Annual Income (k$) and Spending Score (1–100).

In [ ]:
df = pd.read_csv(
    r"C:\Users\staff\Downloads\archive\Mall_Customers.csv"
)

print("First five rows of the dataset:")
print(df.head())

print("\nDataset shape:", df.shape)

print("\nColumn names:", df.columns.tolist())

## 3. Feature Selection & Scaling

**Aim:** Select clustering features and standardise them.

**Objective:** Use Annual Income and Spending Score, then apply StandardScaler.

**Theory:** K-Means uses distances, so features on different scales would bias the result. Standardisation gives each feature equal influence.

In [ ]:
features = [
    "Annual Income (k$)",
    "Spending Score (1-100)"
]

if not all(col in df.columns for col in features):
    raise ValueError(
        "Required columns not found. Check the CSV column names."
    )

X = df[features].apply(pd.to_numeric, errors="coerce")
X = X.dropna()

print("\nNumber of records used:", len(X))
print(X.head())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Feature standardization completed!")
print("Scaled data shape:", X_scaled.shape)

## 4. Find Optimal K

**Aim:** Choose the best number of clusters.

**Objective:** Compute WCSS and Silhouette score for K = 2 to 10.

**Theory:** Lower WCSS means tighter clusters but always falls as K grows, so the 'elbow' and the highest Silhouette score are used to pick K.

In [ ]:
wcss = []
silhouette_scores = []

k_values = range(2, 11)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled)

    wcss.append(kmeans.inertia_)

    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

    print(
        f"K = {k}, WCSS = {kmeans.inertia_:.2f}, "
        f"Silhouette Score = {score:.4f}"
    )

## 5. Elbow Method Plot

**Aim:** Visualise WCSS against K.

**Objective:** Plot the Elbow curve.

**Theory:** The point where the curve bends sharply (the elbow) suggests a good K, since adding more clusters gives little improvement.

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    list(k_values),
    wcss,
    marker="o"
)

plt.title("Elbow Method for Optimal K")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.xticks(list(k_values))
plt.grid(True)
plt.tight_layout()
plt.show()

## 6. Select Optimal K

**Aim:** Choose K using the Silhouette score.

**Objective:** Pick the K with the highest Silhouette score.

**Theory:** A higher Silhouette score means better-separated, more compact clusters.

In [ ]:
best_index = np.argmax(silhouette_scores)
optimal_k = list(k_values)[best_index]

print("\nOptimal K based on Silhouette Score:", optimal_k)

print(
    "Best Silhouette Score:",
    round(silhouette_scores[best_index], 4)
)

## 7. Train Final K-Means Model

**Aim:** Cluster customers with the optimal K.

**Objective:** Fit K-Means and assign cluster labels.

**Theory:** Each customer is assigned to the nearest of the K centroids.

In [ ]:
final_model = KMeans(
    n_clusters=optimal_k,
    random_state=42,
    n_init=10
)

cluster_labels = final_model.fit_predict(X_scaled)

print("Final K-Means model trained successfully!")

result = X.copy()
result["Cluster"] = cluster_labels

print("Cluster assignments:")
print(result.head(10))

print("\nNumber of customers in each cluster:")
print(result["Cluster"].value_counts().sort_index())

## 8. Visualise Clusters

**Aim:** Plot customer segments and centroids.

**Objective:** Draw a scatter plot coloured by cluster with centroids in original units.

**Theory:** Centroids are inverse-transformed to original units so they can be read directly on the income/spending axes.

In [ ]:
centroids_scaled = final_model.cluster_centers_

centroids_original = scaler.inverse_transform(
    centroids_scaled
)

plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    X.iloc[:, 0],
    X.iloc[:, 1],
    c=cluster_labels,
    cmap="viridis",
    s=50,
    alpha=0.8
)

plt.scatter(
    centroids_original[:, 0],
    centroids_original[:, 1],
    marker="X",
    s=250,
    c="red",
    edgecolors="black",
    label="Centroids"
)

plt.title(
    f"K-Means Customer Segmentation (K={optimal_k})"
)

plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend()
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()

## 9. Cluster Profiling & Saving

**Aim:** Interpret and export the segments.

**Objective:** Compute the average income and spending per cluster and save the results.

**Theory:** Cluster averages describe each segment, e.g. high income + high spending = premium customers.

In [ ]:
print("\nCluster-wise feature averages:")

print(
    result.groupby("Cluster")[features]
    .mean()
    .round(2)
)

In [ ]:
result.to_csv(
    r"C:\Users\staff\Downloads\archive\KMeans_Cluster_Results.csv",
    index=False
)

print("\nResults saved successfully!")
print(
    r"File location: C:\Users\staff\Downloads\archive\KMeans_Cluster_Results.csv"
)

## Conclusion

K = 5 gave the best Silhouette score (0.5547) and matches the elbow. The segments are: average income/average spending, high income/high spending, low income/high spending, high income/low spending and low income/low spending.